### proof that in general, longer recording leads to more synapses, and that parts of recording doesn't obscure each other in detecting a monosynapse from time lag

In [ ]:
%load_ext autoreload
%autoreload 2
from neuropy.analyses import correlations 
import numpy as np
import neuropy.analyses.ms_connectivity as msconn
import seaborn as sns
import matplotlib.pyplot as plt
import time
# get only sleep sessions
#### from sd_figure1_bs.ipynb
import subjects
# for i in range(len(subjects.nsd.allsess)):
#     print(subjects.nsd.allsess[i].paradigm)
# subjects.data_table(subjects.nsd.allsess)

# short_neurons = neurons.time_slice(t_start=10000,t_stop=10050)

skips = {} # epoch='post'
skips[msconn.Key(session='RatU_Day2NSD',epoch="post",excitability='E',conn_type=('pyr','pyr'))]=np.array([41,98])
skips[msconn.Key(session='RatU_Day2NSD',epoch="post",excitability='E',conn_type=('pyr','inter'))]=np.array([44,21])
skips[msconn.Key(session='RatU_Day2NSD',epoch="post",excitability='I',conn_type=('inter','inter'))]=np.array([[129,125],[21,0],[139,66]])

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
sess = subjects.nsd.allsess[5]

CCG on the whole session

In [ ]:
# ndconf=msconn.NeuronsDatasetConfig(seg_stride=60*60*0.5, seg_len=60*60*5, recinfo=sess.recinfo,zero_spike_times=False)
ndconf=msconn.NeuronsDatasetConfig(recinfo=sess.recinfo,zero_spike_times=False,epochs=None)
nd_n=msconn.NeuronsDataset(sess,ndconf)
cconf=msconn.CCGConfig(use_acceleration=False,normalize=msconn.NormalizeBy.BOTH_FRATE)
# get maximum number of significant pairs by using the whole dataset
ccg_n=msconn.CCGDataset(nd_n,cconf)
ccg_n.get_ccg()
ccg_n.set_connection_strengths(method="eran_conv")
ccg_n.set_connectivity()
all_inds={}
for EI,v in cconf.conn_types.items():
    for conn_type in v:
        all_inds[conn_type]=ccg_n.data[msconn.Key(session='RatU_Day2NSD',excitability=EI,conn_type=conn_type)].inds

EranConv significant pairs
running eranconv (1st pass)
running spike correlations
spike correlation done
eranconv (1st pass) done
=======RatU_Day2NSD-None=======
Segment(s) are 13.47h each pyr=174 inter=14 
SLEEP0: E/I pairs 435 / 482 | pyr-pyr/E 22 | pyr-inter/E 249 | inter-inter/I 76 | inter-pyr/I 103 | 



CCG by epoch

In [ ]:
# ndconf=msconn.NeuronsDatasetConfig(seg_stride=60*60*0.5, seg_len=60*60*5, recinfo=sess.recinfo,zero_spike_times=False)
epochs=['pre','maze','post','re-maze']
ndconf=msconn.NeuronsDatasetConfig(recinfo=sess.recinfo,zero_spike_times=False,epochs=epochs)
nd=msconn.NeuronsDataset(sess,ndconf)
cconf=msconn.CCGConfig(use_acceleration=False,normalize=msconn.NormalizeBy.BOTH_FRATE)
# get maximum number of significant pairs by using the whole dataset
ccg=msconn.CCGDataset(nd,cconf)
ccg.get_ccg()
ccg.set_connection_strengths(method="eran_conv")
ccg.set_connectivity()
from collections import defaultdict
epoch_inds=defaultdict(lambda: defaultdict(list))
for k,v in ccg.data.items():
    epoch_inds[k.epoch][k.conn_type]=v.inds if v else None

EranConv significant pairs
running eranconv (1st pass)
running spike correlations
spike correlation done
eranconv (1st pass) done
=======RatU_Day2NSD-pre=======
Segment(s) are 2.65h each pyr=174 inter=14 
SLEEP0: E/I pairs 166 / 158 | pyr-pyr/E 06 | pyr-inter/E 91 | inter-inter/I 46 | inter-pyr/I 23 | 

running eranconv (1st pass)
running spike correlations
spike correlation done
eranconv (1st pass) done
=======RatU_Day2NSD-maze=======
Segment(s) are 0.92h each pyr=174 inter=14 
SLEEP0: E/I pairs 068 / 044 | pyr-pyr/E 03 | pyr-inter/E 32 | inter-inter/I 22 | inter-pyr/I 04 | 

running eranconv (1st pass)
running spike correlations
spike correlation done
eranconv (1st pass) done
=======RatU_Day2NSD-post=======
Segment(s) are 9.02h each pyr=174 inter=14 
SLEEP0: E/I pairs 298 / 314 | pyr-pyr/E 15 | pyr-inter/E 172 | inter-inter/I 62 | inter-pyr/I 58 | 

running eranconv (1st pass)
running spike correlations
spike correlation done
eranconv (1st pass) done
=======RatU_Day2NSD-re-maze======

Examine overlap of detected pairs

In [ ]:
RAVEL_SIZE = 200
all_indices_ever={}
for conn_type in cconf.conn_types_flat:
    all_indices_ever[conn_type]=None

df=sess.paradigm.to_dataframe()
df.loc[:, df.columns != 'label'] /= 3600
df = df.round(2)

for e in epochs:
    e_inds=epoch_inds[e]
    text = [' both ', f' {e} only ', f' whole only ']
    print(f"epoch = {e:6} {df.loc[df['label'] == e, 'duration'].values[0]}h | {text[0]} | {text[1]} | {text[2]} |")
    for EI,v in cconf.conn_types.items():
        for conn_type in v:
            e,all=e_inds[conn_type],all_inds[conn_type]
            e_but_not_all = msconn.EranConv.setdiff(e,all,RAVEL_SIZE) 
            all_but_not_e = msconn.EranConv.setdiff(all,e,RAVEL_SIZE)
            all_and_e = msconn.EranConv.intersect(e,all,RAVEL_SIZE)
            example_ind = f'{e_but_not_all[0]}'if e_but_not_all is not None and len(e_but_not_all)>0 and len(e_but_not_all)<=3 else ''
            print(f"{str(conn_type):20} | {str(all_and_e.shape[0]):{len(text[0])}} | {str(e_but_not_all.shape[0])+example_ind:{len(text[1])+1}}| {str(all_but_not_e.shape[0]):{len(text[2])}} |")
            all_indices_ever[conn_type] = msconn.EranConv.union(all_indices_ever[conn_type],msconn.EranConv.union(e,all,RAVEL_SIZE),RAVEL_SIZE)

epoch = pre    2.65h |  both  |  pre only  |  whole only  |
('pyr', 'pyr')       | 5      | 1[97 98]   | 17           |
('pyr', 'inter')     | 88     | 3[ 41 137] | 161          |
('inter', 'inter')   | 46     | 0          | 30           |
('inter', 'pyr')     | 22     | 1[129 160] | 81           |
epoch = maze   0.92h |  both  |  maze only  |  whole only  |
('pyr', 'pyr')       | 2      | 1[179 126]  | 20           |
('pyr', 'inter')     | 32     | 0           | 217          |
('inter', 'inter')   | 22     | 0           | 54           |
('inter', 'pyr')     | 4      | 0           | 99           |
epoch = post   9.02h |  both  |  post only  |  whole only  |
('pyr', 'pyr')       | 14     | 1[88 46]    | 8            |
('pyr', 'inter')     | 171    | 1[187 129]  | 78           |
('inter', 'inter')   | 62     | 0           | 14           |
('inter', 'pyr')     | 58     | 0           | 45           |
epoch = re-maze 0.88h |  both  |  re-maze only  |  whole only  |
('pyr', 'pyr')       | 0 

In [ ]:
all_indices_ever[('pyr','pyr')]

array([[ 23,   9],
       [ 23,  97],
       [ 23, 182],
       [ 26, 157],
       [ 28, 157],
       [ 41,  98],
       [ 52,  41],
       [ 62,  31],
       [ 90,  46],
       [ 97,   9],
       [110,   9],
       [117,  46],
       [120,  90],
       [122,  71],
       [123,  31],
       [147,   7],
       [160,  46],
       [162,  14],
       [165,  46],
       [178,   9],
       [178,  97],
       [182,  10]])

CCG by epoch and segmented

In [ ]:
epochs=['pre','maze','post','re-maze']
ndconf=msconn.NeuronsDatasetConfig(seg_stride=60*60*0.5, seg_len=60*60*5, recinfo=sess.recinfo,zero_spike_times=False,epochs=epochs)
nd_2=msconn.NeuronsDataset(sess,ndconf)
cconf=msconn.CCGConfig(use_acceleration=False,normalize=msconn.NormalizeBy.BOTH_FRATE)
# get maximum number of significant pairs by using the whole dataset
ccg_2=msconn.CCGDataset(nd_2,cconf)
ccg_2.get_ccg()
ccg_2.set_connection_strengths(method="eran_conv")
ccg_2.set_connectivity()

segment length overflowed, using maximum recording time 9544
segment length overflowed, using maximum recording time 3310
segment length overflowed, using maximum recording time 3179
EranConv significant pairs
running eranconv (1st pass)
running spike correlations
spike correlation done
eranconv (1st pass) done
=======RatU_Day2NSD-pre=======
Segment(s) are 2.65h each pyr=174 inter=14 
SLEEP0: E/I pairs 166 / 158 | pyr-pyr/E 06 | pyr-inter/E 91 | inter-inter/I 46 | inter-pyr/I 23 | 

running eranconv (1st pass)
running spike correlations
spike correlation done
eranconv (1st pass) done
=======RatU_Day2NSD-maze=======
Segment(s) are 0.92h each pyr=174 inter=14 
SLEEP0: E/I pairs 068 / 044 | pyr-pyr/E 03 | pyr-inter/E 32 | inter-inter/I 22 | inter-pyr/I 04 | 

running eranconv (1st pass)
running spike correlations
spike correlation done
eranconv (1st pass) done
=======RatU_Day2NSD-post=======
Segment(s) are 5.00h each pyr=174 inter=14 
SLEEP0: E/I pairs 173 / 186 | pyr-pyr/E 07 | pyr-inter

In [ ]:
ccg_2.reCCG_connectivity(inds_by_type=all_indices_ever)
ccg_2.set_connection_strengths()
ccg_2.set_connectivity()

running eranconv (2nd pass): sess_RatU_Day2NSD.epoch_pre.ex_E.type_pyr-pyr
running spike correlations
spike correlation done
done
running eranconv (2nd pass): sess_RatU_Day2NSD.epoch_pre.ex_E.type_pyr-inter
running spike correlations
spike correlation done
done
running eranconv (2nd pass): sess_RatU_Day2NSD.epoch_pre.ex_I.type_inter-inter
running spike correlations
spike correlation done
done
running eranconv (2nd pass): sess_RatU_Day2NSD.epoch_pre.ex_I.type_inter-pyr
running spike correlations
spike correlation done
done
running eranconv (2nd pass): sess_RatU_Day2NSD.epoch_maze.ex_E.type_pyr-pyr
running spike correlations
spike correlation done
done
running eranconv (2nd pass): sess_RatU_Day2NSD.epoch_maze.ex_E.type_pyr-inter
running spike correlations
spike correlation done
done
running eranconv (2nd pass): sess_RatU_Day2NSD.epoch_maze.ex_I.type_inter-inter
running spike correlations
spike correlation done
done
running eranconv (2nd pass): sess_RatU_Day2NSD.epoch_maze.ex_I.type_inter

In [ ]:
ccg_2.save_plots(conn_types=[('pyr','pyr')],root="~/Documents/NeuroPy/images/ccg_plots_test0109")

[Key(session='RatU_Day2NSD', epoch='pre', ref_ind=None, target_ind=None, segment=None, excitability='E', conn_type=('pyr', 'pyr')), Key(session='RatU_Day2NSD', epoch='maze', ref_ind=None, target_ind=None, segment=None, excitability='E', conn_type=('pyr', 'pyr')), Key(session='RatU_Day2NSD', epoch='post', ref_ind=None, target_ind=None, segment=None, excitability='E', conn_type=('pyr', 'pyr'))]
Saving plots under ~/Documents/NeuroPy/images/ccg_plots_test0109
ccg RatU_Day2NSD ('pyr', 'pyr')
0 [23  9]
1 [23 97]
2 [ 23 182]
3 [ 26 157]
4 [ 28 157]
5 [41 98]
6 [52 41]
7 [62 31]
8 [88 46]
9 [90 46]
10 [97  9]
11 [97 98]
12 [110   9]
13 [117  46]
14 [120  90]
15 [122  71]
16 [123  31]
17 [147   7]
18 [160  46]
19 [162  14]
20 [165  46]
21 [178   9]
22 [178  97]
23 [179 126]
24 [182  10]
done saving plots
ccg RatU_Day2NSD ('pyr', 'pyr')
0 [23  9]
1 [23 97]
2 [ 23 182]
3 [ 26 157]
4 [ 28 157]
5 [41 98]
6 [52 41]
7 [62 31]
8 [88 46]
9 [90 46]
10 [97  9]
11 [97 98]
12 [110   9]
13 [117  46]
14 [12